<a href="https://colab.research.google.com/github/ben-blake/umkc-deep-learning/blob/main/hw7-network-compression/HW7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Homework 7 - Network Compression

**Name:** Ben Blake

**Student ID:** 14387365

## **Intro**

HW7 is about network compression

There are many types of Network/Model Compression,  here we introduce two:
* Knowledge Distillation
* Design Architecture


The process of this notebook is as follows: <br/>
1. Introduce depthwise, pointwise and group convolution in MobileNet.
2. Design the model of this colab
3. Introduce Knowledge-Distillation
4. Set up TeacherNet and it would be helpful in training


## **About the Dataset**

The dataset used here is food-11, a collection of food images in 11 classes.

For the requirement in the homework，the network is slightly different from the original version so please DO NOT access the original fully-labeled training data or testing labels.


In [1]:
# Download food-11.zip from share files and put it in your own GoogleDrive
# The following code allows colab to access your GoogleDrive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Unzip the dataset.
# This may take some time.
# Put the pretrained resnet_food11 in your OWN GoogleDrive, the path need to specify!!
PATH = '/content/drive/My Drive/UMKC/Fall 2026/Deep Learning/HW7/ResNet_food11.pth'
# food-11.zip and the teacher checkpoint both live in the HW7 folder on Drive. unzips to ./food-11
!unzip -q -o "/content/drive/My Drive/UMKC/Fall 2026/Deep Learning/HW7/food-11.zip" -d /content/

## **Import Packages**

First, we need to import packages that will be used later.

In this homework, we highly rely on **torchvision**, a library of PyTorch.

In [3]:
# Import necessary packages.
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch
import torchvision.transforms as transforms
import torchvision.models as models

from PIL import Image
# "ConcatDataset" and "Subset" are possibly useful when doing semi-supervised learning.
from torch.utils.data import ConcatDataset, DataLoader, Subset
from torchvision.datasets import DatasetFolder

# This is for the progress bar.
# from tqdm.auto import tqdm
from tqdm import tqdm

## **Dataset, Data Loader, and Transforms** (2 pts)

Torchvision provides lots of useful utilities for image preprocessing, data wrapping as well as data augmentation.

Here, since our data are stored in folders by class labels, we can directly apply **torchvision.datasets.DatasetFolder** for wrapping data without much effort. You can refer to [PyTorch official website](https://pytorch.org/vision/stable/transforms.html) for details about different transforms.


In [4]:
# It is important to do data augmentation in training.
# However, not every augmentation is useful.
# Please think about what kind of augmentation is helpful for food recognition.

train_tfm = transforms.Compose([
  # Resize the image into a fixed shape (height = width = 142)
	transforms.Resize((142, 142)),
  # (2 pts) TODO: Apply in order: RandomHorizontalFilp, RandomRotation(15), RandomCrop to size 128, and convert to Tensor
  transforms.RandomHorizontalFlip(),
  transforms.RandomRotation(15),
  transforms.RandomCrop(128),
  transforms.ToTensor(),
  # a flipped or slightly tilted plate of food is still the same dish, so these don't change the label.
  # the random 128 crop out of 142 matches the CenterCrop(128) in test_tfm, so train and test inputs are the same size
])

# We don't need augmentations in testing and validation.
# All we need here is to resize the PIL image and transform it into Tensor.
test_tfm = transforms.Compose([
    # Resize the image into a fixed shape (height = width = 142)
    transforms.Resize((142, 142)),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
])


In [5]:
# Batch size for training, validation, and testing.
# A greater batch size usually gives a more stable gradient.
# But the GPU memory is limited, so please adjust it carefully.
batch_size = 64

# Construct datasets.
# The argument "loader" tells how torchvision reads the data.
train_set = DatasetFolder("food-11/training/labeled", loader=lambda x: Image.open(x), extensions="jpg", transform=train_tfm)
valid_set = DatasetFolder("food-11/validation", loader=lambda x: Image.open(x), extensions="jpg", transform=test_tfm)
unlabeled_set = DatasetFolder("food-11/training/unlabeled", loader=lambda x: Image.open(x), extensions="jpg", transform=train_tfm)
test_set = DatasetFolder("food-11/testing", loader=lambda x: Image.open(x), extensions="jpg", transform=test_tfm)

# Construct data loaders.
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
# num_workers raised from 0 to 2 (the T4 runtime has 2 CPUs) so jpg decoding and augmentation run in parallel
# with the GPU. with 0 workers the GPU sat idle most of each epoch
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

# **Architecture / Model Design**
The following are types of convolution layer design that has fewer parameters.

## **Depthwise & Pointwise Convolution**
![](https://i.imgur.com/FBgcA0s.png)
> Blue: the connection between layers \
> Green: the expansion of **receptive field** \
> (reference: arxiv:1810.04231)

(a) normal convolution layer: It is fully connected. The difference between fully connected layer and fully connected convolution layer is the operation. (multiply --> convolution)

(b) Depthwise convolution layer(DW): You can consider each feature map pass through their own filter and then pass through pointwise convolution layer(PW) to combine the information of all pixels in feature maps.


(c) Group convolution layer(GC): Group the feature maps. Each group passes their filter then concate together. If group_size = input_feature_size, then GC becomes DC (channels are independent). If group_size = 1, then GC becomes fully connected.

<img src="https://i.imgur.com/Hqhg0Q9.png" width="500px">


## **Implementation details**
```python
# Regular Convolution, # of params = in_chs * out_chs * kernel_size^2
nn.Conv2d(in_chs, out_chs, kernel_size, stride, padding)

# Group Convolution, "groups" controls the connections between inputs and
# outputs. in_chs and out_chs must both be divisible by groups.
nn.Conv2d(in_chs, out_chs, kernel_size, stride, padding, groups=groups)

# Depthwise Convolution, out_chs=in_chs=groups, # of params = in_chs * kernel_size^2
nn.Conv2d(in_chs, out_chs=in_chs, kernel_size, stride, padding, groups=in_chs)

# Pointwise Convolution, a.k.a 1 by 1 convolution, # of params = in_chs * out_chs
nn.Conv2d(in_chs, out_chs, 1)

# Merge Depthwise and Pointwise Convolution (without )
def dwpw_conv(in_chs, out_chs, kernel_size, stride, padding):
    return nn.Sequential(
        nn.Conv2d(in_chs, in_chs, kernels, stride, padding, groups=in_chs),
        nn.Conv2d(in_chs, out_chs, 1),
    )
```

## **Model**

The basic model here is simply a stack of convolutional layers followed by some fully-connected layers. You can take advatage of depthwise & pointwise convolution to make your model deeper, but still follow the size constraint.

In [6]:
class StudentNet(nn.Module):
    def __init__(self):
      super(StudentNet, self).__init__()

      self.cnn = nn.Sequential(
        nn.Conv2d(3, 32, 3),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.Conv2d(32, 32, 3),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2, 2, 0),

        nn.Conv2d(32, 64, 3),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2, 2, 0),

        nn.Conv2d(64, 100, 3),
        nn.BatchNorm2d(100),
        nn.ReLU(),
        nn.MaxPool2d(2, 2, 0),

        # Here we adopt Global Average Pooling for various input size.
        nn.AdaptiveAvgPool2d((1, 1)),
      )

      # (2 pts) TODO: Apply fully connected layer with correct input and output channels
      self.fc = nn.Sequential(
        nn.Linear(100, 11),
        # the global average pool leaves 100 channels at 1x1, so flattening gives 100 features, and food-11 has 11 classes.
        # no softmax here since cross_entropy and the KD loss both expect raw logits
      )

    def forward(self, x):
      out = self.cnn(x)
      out = out.view(out.size()[0], -1)
      return self.fc(out)


## **Model Analysis**

Use `torchsummary` to get your model architecture (screenshot or pasting text are allowed.) and numbers of
parameters, these two information should be submit.
Note that the number of parameters **should not greater than 100,000**


In [7]:
from torchsummary import summary

student_net = StudentNet()
summary(student_net, (3, 128, 128), device="cpu")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 126, 126]             896
       BatchNorm2d-2         [-1, 32, 126, 126]              64
              ReLU-3         [-1, 32, 126, 126]               0
            Conv2d-4         [-1, 32, 124, 124]           9,248
       BatchNorm2d-5         [-1, 32, 124, 124]              64
              ReLU-6         [-1, 32, 124, 124]               0
         MaxPool2d-7           [-1, 32, 62, 62]               0
            Conv2d-8           [-1, 64, 60, 60]          18,496
       BatchNorm2d-9           [-1, 64, 60, 60]             128
             ReLU-10           [-1, 64, 60, 60]               0
        MaxPool2d-11           [-1, 64, 30, 30]               0
           Conv2d-12          [-1, 100, 28, 28]          57,700
      BatchNorm2d-13          [-1, 100, 28, 28]             200
             ReLU-14          [-1, 100,

## **Knowledge Distillation**

<img src="https://i.imgur.com/H2aF7Rv.png=100x" width="500px">

Since we have a learned big model, let it teach the other small model. In implementation, let the training target be the prediction of big model instead of the ground truth.

## **Why it works?**
* If the data is not clean, then the prediction of big model could ignore the noise of the data with wrong labeled.
* The labels might have some relations. Number 8 is more similar to 6, 9, 0 than 1, 7, for example.


## **How to implement?**
* $Loss = \alpha T^2 \times KL(\frac{\text{Teacher's Logits}}{T} || \frac{\text{Student's Logits}}{T}) + (1-\alpha)(\text{Original Loss})$
* Note that the logits here should have passed softmax.

In [8]:
def loss_fn_kd(outputs, labels, teacher_outputs, alpha=0.5):
    hard_loss = F.cross_entropy(outputs, labels) * (1. - alpha)
    # ---------- TODO ----------
    # Complete soft loss in knowledge distillation (The KL term for the soft loss, if you put soft_loss=0, you will loss this term instead)
    T = 20
    soft_loss = F.kl_div(F.log_softmax(outputs / T, dim=1), F.softmax(teacher_outputs / T, dim=1), reduction='batchmean') * (alpha * T * T)
    # KL(teacher || student) on temperature-softened distributions. F.kl_div wants the student as log-probs and the
    # teacher as plain probs, and batchmean averages over the batch so the scale matches cross_entropy.
    # a high T flattens the teacher's output so the small probabilities on the "wrong but similar" classes actually matter.
    # softening shrinks the gradients by about 1/T^2, so multiplying by T^2 keeps the soft term on the same scale as the hard one
    return hard_loss + soft_loss

## **Teacher Model Setting**
We provide a well-trained teacher model to help you knowledge distillation to student model.
Note that if you want to change the transform function, you should consider  if suitable for this well-trained teacher model.
* If you cannot successfully gdown, you can change a link. (Backup link is provided at the bottom of this colab tutorial).


In [9]:
# Download teacherNet
import torchvision.models as models
# Load teacherNet
teacher_net = models.resnet18(pretrained=False)
teacher_net.fc = nn.Linear(in_features=512, out_features=11, bias=True)
checkpoint = torch.load(PATH)
teacher_net.load_state_dict(checkpoint)
teacher_net.eval()

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

## **Generate Pseudo Labels in Unlabeled Data**

Since we have a well-trained model, we can use this model to predict pseudo-labels and help the student network train well. Note that you
**CANNOT** use well-trained model to pseudo-label the test data.


---

**AGAIN, DO NOT USE TEST DATA FOR PURPOSE OTHER THAN INFERENCING**

* Because If you use teacher network to predict pseudo-labels of the test data, you can only use student network to overfit these pseudo-labels without train/unlabeled data. In this way, your kaggle accuracy will be as high as the teacher network, but the fact is that you just overfit the test data and your true testing accuracy is very low.
* These contradict the purpose of these assignment (network compression); therefore, you should not misuse the test data.


In [10]:
# "cuda" only when GPUs are available.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

# Initialize a model, and put it on the device specified.
student_net = student_net.to(device)
teacher_net = teacher_net.to(device)

# Whether to do pseudo label.
do_semi = True

def get_pseudo_labels(dataset, model):
    loader = DataLoader(dataset, batch_size=batch_size*3, shuffle=False, pin_memory=True, num_workers=2)
    pseudo_labels = []
    for batch in tqdm(loader):
        # A batch consists of image data and corresponding labels.
        img, _ = batch

        # Forward the data

        with torch.no_grad():
            logits = model(img.to(device))
            pseudo_labels.append(logits.argmax(dim=-1).detach().cpu())
        # Obtain the probability distributions by applying softmax on logits.
    pseudo_labels = torch.cat(pseudo_labels)
    # Update the labels by replacing with pseudo labels.
    for idx, ((img, _), pseudo_label) in enumerate(zip(dataset.samples, pseudo_labels)):
        dataset.samples[idx] = (img, pseudo_label.item())
    return dataset

if do_semi:
    # Generate new trainloader with unlabeled set.
    unlabeled_set = get_pseudo_labels(unlabeled_set, teacher_net)
    print(unlabeled_set)
    concat_dataset = ConcatDataset([train_set, unlabeled_set])
    train_loader = DataLoader(concat_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, drop_last=True, num_workers=2)
    # same num_workers change as the loaders above, this is the loader the training loop actually uses




cuda


100%|██████████| 36/36 [00:27<00:00,  1.32it/s]

Dataset DatasetFolder
    Number of datapoints: 6786
    Root location: food-11/training/unlabeled
    StandardTransform
Transform: Compose(
               Resize(size=(142, 142), interpolation=bilinear, max_size=None, antialias=True)
               RandomHorizontalFlip(p=0.5)
               RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
               RandomCrop(size=(128, 128), padding=None)
               ToTensor()
           )


## **Training** (*14* pts)

You can finish supervised learning by simply running the provided code without any modification.

The function "get_pseudo_labels" is used for semi-supervised learning.
It is expected to get better performance if you use unlabeled data for semi-supervised learning.
However, you have to implement the function on your own and need to adjust several hyperparameters manually.
Again, please notice that utilizing external data (or pre-trained model) for training is **prohibited**.

---
**You should use loss in  knowledge distillation.**




In [11]:
# For the classification task, we use cross-entropy as the measurement of performance.
# (2 pts) TODO: Apply crossentropy loss
criterion = nn.CrossEntropyLoss()
# standard multi-class loss on raw logits. the loop below trains on loss_fn_kd, whose hard term is this same
# cross entropy (through F.cross_entropy), so criterion itself doesn't get called

# (3 pts) TODO: Set your own optimizer(Hint: Aplly Adam from torch.optim to student_net.parameters with lr=0.0003, weight_decay=1e-5)
optimizer = torch.optim.Adam(student_net.parameters(), lr=0.0003, weight_decay=1e-5)
# only the student's parameters go in. the teacher stays frozen the whole time

# The number of training epochs.
n_epochs = 30
# lowered from 80. the Colab runtime kept disconnecting partway through the longer runs, so this keeps training to ~12 minutes

for epoch in range(n_epochs):
    # ---------- Training ----------

    # (1pt) TODO: Set the model into training mode
    student_net.train()
    # puts BatchNorm back on per-batch statistics, since validation at the end of each epoch switched it to eval

    # These are used to record information in training.
    train_loss = []
    train_accs = []

    # Iterate the training set by batches.
    for batch in tqdm(train_loader):

        # A batch consists of image data and corresponding labels.
        imgs, labels = batch

        # Forward the data. (Make sure data and model are on the same device.
        logits = student_net(imgs.to(device))
        # Teacher net will not be updated. And we use torch.no_grad
        # to tell torch do not retain the intermediate values
        # (which are for backpropgation) and save the memory.

        with torch.no_grad():
          # (4 pts) TODO: put imgs to teacher_net to create soft labels(Remember to put imgs onto gpu)
          soft_labels = teacher_net(imgs.to(device))
          # the teacher sees the exact same augmented batch as the student, so its logits are a target for these
          # specific images. teacher_net is still in eval mode from the loading cell, so its BatchNorm stats stay fixed

        # Calculate the loss in knowledge distillation method.
        loss = loss_fn_kd(logits, labels.to(device), soft_labels)

        # Gradients stored in the parameters in the previous step should be cleared out first.
        optimizer.zero_grad()

        # Compute the gradients for parameters.
        loss.backward()

        # (2 pts) TODO: Apply nn.utils.clip_grad_norm_ for stable training (max_norm=10)
        grad_norm = nn.utils.clip_grad_norm_(student_net.parameters(), max_norm=10)
        # rescales the gradients if their combined L2 norm is over 10, and returns the norm from before clipping.
        # the T^2 factor in the soft loss can make early gradients large, so this keeps the first few steps from blowing up

        # Update the parameters with computed gradients.
        optimizer.step()

        # Compute the accuracy for current batch.
        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()

        # Record the loss and accuracy.
        train_loss.append(loss.item())
        train_accs.append(acc)

    # The average loss and accuracy of the training set is the average of the recorded values.
    train_loss = sum(train_loss) / len(train_loss)
    train_acc = sum(train_accs) / len(train_accs)

    # Print the information.
    print(f"[ Train | {epoch + 1:03d}/{n_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")


    # ---------- Validation ----------
    # (1pt)TODO: Set the model into eval mode so that some modules like dropout are disabled and work normally.
    student_net.eval()
    # the student has no dropout, but it does have BatchNorm, which now uses the running mean/var from training

    # These are used to record information in validation.
    valid_loss = []
    valid_accs = []

    # Iterate the validation set by batches.
    for batch in tqdm(valid_loader):

        # A batch consists of image data and corresponding labels.
        imgs, labels = batch

        # We don't need gradient in validation.
        # (1 pt) TODO: Using torch.no_grad() accelerates the forward process.
        with torch.no_grad():
          # no graph gets built for either forward pass, which saves memory and time since nothing is backpropagated here
          logits = student_net(imgs.to(device))
          soft_labels = teacher_net(imgs.to(device))
        # We can still compute the loss (but not the gradient).
        loss = loss_fn_kd(logits, labels.to(device), soft_labels)

        # Compute the accuracy for current batch.
        acc = (logits.argmax(dim=-1) == labels.to(device)).float().detach().cpu().view(-1).numpy()

        # Record the loss and accuracy.
        valid_loss.append(loss.item())
        valid_accs += list(acc)

    # The average loss and accuracy for entire validation set is the average of the recorded values.
    valid_loss = sum(valid_loss) / len(valid_loss)
    valid_acc = sum(valid_accs) / len(valid_accs)

    # Print the information.
    print(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")

100%|██████████| 154/154 [00:43<00:00,  3.57it/s]


[ Train | 001/030 ] loss = 5.43928, acc = 0.31250


100%|██████████| 11/11 [00:03<00:00,  3.47it/s]


[ Valid | 001/030 ] loss = 5.08578, acc = 0.23636


100%|██████████| 154/154 [00:42<00:00,  3.65it/s]


[ Train | 002/030 ] loss = 5.01703, acc = 0.37652


... (110 outputs left out)

100%|██████████| 154/154 [00:41<00:00,  3.68it/s]


[ Train | 030/030 ] loss = 3.35526, acc = 0.62094


100%|██████████| 11/11 [00:02<00:00,  3.73it/s]

[ Valid | 030/030 ] loss = 3.51857, acc = 0.53939


## **Testing**

For inference, we need to make sure the model is in eval mode, and the order of the dataset should not be shuffled ("shuffle=False" in test_loader).


In [12]:
# Make sure the model is in eval mode.
# Some modules like Dropout or BatchNorm affect if the model is in training mode.
student_net.eval()

# Initialize a list to store the predictions.
predictions = []

# Iterate the testing set by batches.
for batch in tqdm(test_loader):
    # A batch consists of image data and corresponding labels.
    # But here the variable "labels" is useless since we do not have the ground-truth.
    # If printing out the labels, you will find that it is always 0.
    # This is because the wrapper (DatasetFolder) returns images and labels for each batch,
    # so we have to create fake labels to make it work normally.
    imgs, labels = batch

    # We don't need gradient in testing, and we don't even have labels to compute loss.
    # Using torch.no_grad() accelerates the forward process.
    with torch.no_grad():
        logits = student_net(imgs.to(device))

    # Take the class with greatest logit as prediction and record it.
    predictions.extend(logits.argmax(dim=-1).cpu().numpy().tolist())

100%|██████████| 53/53 [00:17<00:00,  3.08it/s]


In [13]:
# Save predictions into the file.
with open("predict.csv", "w") as f:

    # The first row must be "Id, Category"
    f.write("Id,Category\n")

    # For the rest of the rows, each image id corresponds to a predicted class.
    for i, pred in  enumerate(predictions):
         f.write(f"{i},{pred}\n")

## **Statistics**

|Baseline|Accuracy|Training Time|
|-|-|-|
|Simple Baseline |0.59856|2 Hours|
|Medium Baseline |0.65412|2 Hours|
|Strong Baseline |0.72819|4 Hours|
|Boss Baseline |0.81003|Unmeasueable|

## **Learning Curve**

![img](https://lh5.googleusercontent.com/amMLGa7dkqvXGmsJlrVN49VfSjClk5d-n7nCi_Y3ROK4himsBSHhB7SpdWe7Zm06ctRO77VdDkD9u_aKfAh1tMW-KcyYX7vF7LPlKqOo2fVtt3SyfsLv0KTYDB0YbAk6ZhyOIKT8Zfg)

# **Summary**

I filled in the TODOs and trained the student CNN with knowledge distillation from the provided ResNet-18 teacher on a T4. The student has 87,907 parameters, under the 100,000 limit, and reaches 0.5394 validation accuracy after 30 epochs (best 0.5591 at epoch 27). The teacher gets 0.7955 on the same validation set with about 127x as many parameters (11.2M).

**Data**

food-11 has 3,080 labeled training images, 6,786 unlabeled ones, 660 validation and 3,347 test images over 11 classes. Training images get a random horizontal flip, a random rotation of up to 15 degrees and a random 128x128 crop out of the 142x142 resize. Validation and test just get resized and center cropped to 128. The teacher pseudo-labels the unlabeled set with its argmax prediction, and that gets concatenated with the labeled set, so each epoch is 9,856 images (154 batches of 64). The test set is only used for the final predictions.

**Model**

The student is the template's four conv layers (3 to 32 to 32 to 64 to 100 channels, each with BatchNorm and ReLU, three max pools) followed by global average pooling. That leaves 100 features, so the classifier TODO is `Linear(100, 11)`, which adds 1,111 parameters. Most of the student's size is the last conv layer (57,700).

**Loss**

The soft loss is `F.kl_div` between the temperature-softened student log-probabilities and teacher probabilities, with T = 20, averaged over the batch and scaled by alpha * T^2 (alpha = 0.5). Dividing by T flattens the teacher's output so the student also learns which wrong classes the teacher thinks are close, and the T^2 factor keeps the soft term's gradients on the same scale as the hard cross entropy term. Training used Adam (lr 3e-4, weight decay 1e-5) with gradient norm clipping at 10. The teacher's logits come from the same augmented batch under `torch.no_grad()`, and it stays in eval mode the whole time.

**Results**

| | parameters | valid accuracy |
|---|---|---|
| teacher (ResNet-18) | ~11.2M | 0.7955 |
| student, epoch 30 | 87,907 | 0.5394 |
| student, best (epoch 27) | 87,907 | 0.5591 |

Train accuracy rose steadily from 0.313 to 0.621 and the KD loss fell from 5.44 to 3.36. Validation accuracy went from 0.236 to the mid 0.50s, but it jumps around by up to 0.07 between epochs since there are only 660 images. The gap between train and validation is about 0.08 at the end and wasn't growing, so the student is still underfitting rather than overfitting. Training accuracy here is also measured against the teacher's pseudo-labels for two thirds of the data, so it isn't a clean number.

**Changes outside the TODOs**

- The template trains for 80 epochs. The Colab runtime disconnected twice in the middle of longer runs (once at epoch 55 of 80), so I lowered `n_epochs` to 30 to get a complete run. In an earlier 80-epoch attempt that got to epoch 54 before disconnecting, validation accuracy reached about 0.61 by epochs 50 to 54, so the full 80 epochs would most likely land around there, near the simple baseline.
- `num_workers` in the DataLoaders went from 0 to 2. With 0, all the image decoding and augmentation happened on one thread and the GPU spent most of each epoch waiting. This cut an epoch from about 65 s to 24 to 44 s depending on the runtime and doesn't change what the model sees.
- The Drive paths in the unzip cell point to my HW7 folder for both `food-11.zip` and `ResNet_food11.pth`.

The student is never checkpointed at its best validation epoch, so `predict.csv` comes from the epoch 30 weights.

#**AI Use Disclosure**

Anthropic Claude Code was utilized on this assignment. Its contributions were: assistance with the explanatory comments next to updated TODO items and the summary. All code outside the TODO markers is the provided template, unmodified apart from the Google Drive mount and dataset unzip cells, the `num_workers` setting on the DataLoaders, and `n_epochs`. I personally modified, reviewed, and ran the final notebook.

#**Documentation Referenced**

**Data loading:**
- torchvision.transforms (RandomHorizontalFlip, RandomRotation, RandomCrop, ToTensor) - https://pytorch.org/vision/stable/transforms.html
- torchvision.datasets.DatasetFolder - https://pytorch.org/vision/stable/generated/torchvision.datasets.DatasetFolder.html
- torch.utils.data (DataLoader, ConcatDataset, num_workers) - https://pytorch.org/docs/stable/data.html

**Model:**
- nn.Conv2d - https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
- nn.BatchNorm2d - https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html
- nn.AdaptiveAvgPool2d - https://pytorch.org/docs/stable/generated/torch.nn.AdaptiveAvgPool2d.html
- nn.Linear - https://pytorch.org/docs/stable/generated/torch.nn.Linear.html
- torchvision.models.resnet18 - https://pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html
- torchsummary - https://github.com/sksq96/pytorch-summary
- MobileNets (depthwise separable convolutions) - https://arxiv.org/abs/1704.04861

**Loss & training:**
- F.cross_entropy - https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html
- F.kl_div - https://pytorch.org/docs/stable/generated/torch.nn.functional.kl_div.html
- F.log_softmax - https://pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html
- F.softmax - https://pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html
- nn.CrossEntropyLoss - https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html
- torch.optim.Adam - https://pytorch.org/docs/stable/generated/torch.optim.Adam.html
- nn.utils.clip_grad_norm_ - https://pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html
- torch.no_grad - https://pytorch.org/docs/stable/generated/torch.no_grad.html
- nn.Module.train / nn.Module.eval - https://pytorch.org/docs/stable/generated/torch.nn.Module.html
- Distilling the Knowledge in a Neural Network - https://arxiv.org/abs/1503.02531